dcm文件已经满足：
彩色三通道
像素范围通常为 0~255
适合直接保存为 RGB PNG

转换逻辑为
bmp / jpg / jpeg / png：直接 convert("RGB") 后保存为 png
dcm：直接读取 pixel_array，确认是 RGB uint8 后保存为 png





下面代码
1. 递归读取 batch2data_1\raw 下的全部文件
2. 图像文件统一转为 RGB PNG
3. DICOM 按 RGB uint8 彩色图像处理，不归一化、不改颜色
4. 非图像文件，例如 case_meta.json、visit_meta.json、report.txt、csv 等，直接复制
5. 输出到 batch2data_2\raw
6. 除了图像扩展名统一变成 .png，其他目录层级不变

In [2]:
from pathlib import Path
from PIL import Image
import numpy as np
import pydicom
import shutil

# =========================
# 1. 输入目录和输出目录
# =========================
input_root = Path(r"G:\wxy_codes\Fetal_hydronephrosis\1第一批数据\batch1data_1\raw")
output_root = Path(r"G:\wxy_codes\Fetal_hydronephrosis\1第一批数据\batch1data_2\raw")

# 需要统一转换为 PNG 的图像格式
image_suffixes = {".bmp", ".jpg", ".jpeg", ".png", ".dcm"}


# =========================
# 2. 普通图像读取
# =========================
def read_normal_image_as_rgb(image_path):
    """
    普通图像：直接转为 RGB，不做归一化。
    """
    return Image.open(image_path).convert("RGB")


# =========================
# 3. DICOM 图像读取
# =========================
def read_rgb_uint8_dicom_as_rgb(dcm_path):
    """
    适用于你的当前数据：
    RGB 彩色 DICOM，BitsAllocated=8，pixel_array=uint8。
    直接保留 RGB 三通道，不做归一化。
    """
    ds = pydicom.dcmread(str(dcm_path), force=True)
    arr = ds.pixel_array

    photometric = getattr(ds, "PhotometricInterpretation", "")
    samples_per_pixel = getattr(ds, "SamplesPerPixel", None)

    if photometric != "RGB" or samples_per_pixel != 3:
        raise ValueError(
            f"不是标准 RGB 彩色 DICOM: "
            f"PhotometricInterpretation={photometric}, SamplesPerPixel={samples_per_pixel}"
        )

    if arr.dtype != np.uint8:
        raise ValueError(f"DICOM 像素类型不是 uint8，而是 {arr.dtype}")

    # 多帧 RGB DICOM：默认只取第一帧
    if arr.ndim == 4:
        arr = arr[0]

    if arr.ndim != 3 or arr.shape[-1] != 3:
        raise ValueError(f"DICOM 图像维度异常: {arr.shape}")

    return Image.fromarray(arr, mode="RGB")


# =========================
# 4. 批量转换与复制
# =========================
success_image_count = 0
copy_file_count = 0
fail_count = 0
fail_list = []

if not input_root.exists():
    raise FileNotFoundError(f"输入目录不存在: {input_root}")

output_root.mkdir(parents=True, exist_ok=True)

all_files = [p for p in input_root.rglob("*") if p.is_file()]

print(f"共找到 {len(all_files)} 个文件。")
print(f"输入目录: {input_root}")
print(f"输出目录: {output_root}\n")

for src_path in all_files:
    try:
        relative_path = src_path.relative_to(input_root)
        dst_path = output_root / relative_path
        dst_path.parent.mkdir(parents=True, exist_ok=True)

        suffix = src_path.suffix.lower()

        # 情况1：图像文件，统一转换为 PNG
        if suffix in image_suffixes:
            if suffix == ".dcm":
                img = read_rgb_uint8_dicom_as_rgb(src_path)
            else:
                img = read_normal_image_as_rgb(src_path)

            dst_png_path = dst_path.with_suffix(".png")
            img.save(dst_png_path)
            success_image_count += 1

        # 情况2：非图像文件，原样复制
        else:
            shutil.copy2(src_path, dst_path)
            copy_file_count += 1

    except Exception as e:
        fail_count += 1
        fail_list.append((str(src_path), str(e)))
        print(f"处理失败: {src_path}")
        print(f"原因: {e}\n")

print("\n全部处理完成。")
print(f"成功转换图像文件: {success_image_count} 个")
print(f"成功复制非图像文件: {copy_file_count} 个")
print(f"处理失败文件: {fail_count} 个")

if fail_list:
    print("\n失败文件列表：")
    for path, reason in fail_list:
        print(path)
        print("原因:", reason)
        print()

共找到 2660 个文件。
输入目录: G:\wxy_codes\Fetal_hydronephrosis\1第一批数据\batch1data_1\raw
输出目录: G:\wxy_codes\Fetal_hydronephrosis\1第一批数据\batch1data_2\raw


全部处理完成。
成功转换图像文件: 2660 个
成功复制非图像文件: 0 个
处理失败文件: 0 个


In [1]:
##检查图像位深
from pathlib import Path
from collections import Counter
from PIL import Image
import numpy as np

# 如果需要检查 DICOM，请先安装：
# pip install pydicom
try:
    import pydicom
except ImportError:
    pydicom = None


# =========================
# 1. 设置根目录
# =========================
root_dir = Path(r"G:\wxy_codes\Fetal_hydronephrosis\1第一批数据\batch1data_1\raw")

image_suffixes = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".dcm"}


# =========================
# 2. 判断普通图像位深
# =========================
def get_normal_image_bit_depth(image_path):
    img = Image.open(image_path)
    arr = np.array(img)

    # 每个通道的 bit depth
    if arr.dtype == np.uint8:
        bits_per_channel = 8
    elif arr.dtype == np.uint16:
        bits_per_channel = 16
    elif arr.dtype == np.int16:
        bits_per_channel = 16
    elif arr.dtype == np.float32:
        bits_per_channel = 32
    elif arr.dtype == np.float64:
        bits_per_channel = 64
    else:
        bits_per_channel = f"Unknown dtype={arr.dtype}"

    # 通道数
    if arr.ndim == 2:
        channels = 1
    elif arr.ndim == 3:
        channels = arr.shape[-1]
    else:
        channels = f"Unknown shape={arr.shape}"

    return f"{bits_per_channel}-bit x {channels}通道"


# =========================
# 3. 判断 DICOM 位深
# =========================
def get_dicom_bit_depth(dcm_path):
    if pydicom is None:
        raise ImportError("未安装 pydicom，请先运行：pip install pydicom")

    ds = pydicom.dcmread(str(dcm_path), stop_before_pixels=True, force=True)

    bits_allocated = getattr(ds, "BitsAllocated", "Unknown")
    bits_stored = getattr(ds, "BitsStored", "Unknown")
    samples_per_pixel = getattr(ds, "SamplesPerPixel", "Unknown")
    photometric = getattr(ds, "PhotometricInterpretation", "Unknown")

    return (
        f"BitsAllocated={bits_allocated}, "
        f"BitsStored={bits_stored}, "
        f"SamplesPerPixel={samples_per_pixel}, "
        f"{photometric}"
    )


# =========================
# 4. 递归统计
# =========================
bit_depth_counter = Counter()
fail_count = 0
fail_list = []

image_files = [
    p for p in root_dir.rglob("*")
    if p.is_file() and p.suffix.lower() in image_suffixes
]

print(f"共找到 {len(image_files)} 个图像文件。\n")

for img_path in image_files:
    try:
        suffix = img_path.suffix.lower()

        if suffix == ".dcm":
            bit_depth = get_dicom_bit_depth(img_path)
        else:
            bit_depth = get_normal_image_bit_depth(img_path)

        bit_depth_counter[bit_depth] += 1

    except Exception as e:
        fail_count += 1
        fail_list.append((str(img_path), str(e)))


# =========================
# 5. 输出统计结果
# =========================
print("图像位深统计：\n")

for bit_depth, count in bit_depth_counter.items():
    print(f"{bit_depth}：{count} 张")

print(f"\n读取失败：{fail_count} 张")

if fail_list:
    print("\n失败原因示例：")
    for path, reason in fail_list[:10]:
        print(path)
        print("原因:", reason)
        print()

共找到 2660 个图像文件。

图像位深统计：

BitsAllocated=8, BitsStored=8, SamplesPerPixel=3, RGB：2083 张
8-bit x 3通道：577 张

读取失败：0 张
